# Citebound: Enterprise Hybrid RAG Analyzer for Legal & Financial Documents

# Dependencies & Setup
Installs core project dependencies. The pipeline uses faiss-cpu for lightweight in-memory vector search and implements a fallback import block to ensure compatibility across LangChain 0.3 and 1.x.

In [ ]:
# Quiet installs. Rank_bm25 is tiny; sentence-transformers is the heavy one
# (~80MB for MiniLM). faiss-cpu so we do not pull the CUDA wheel by accident.
%pip -q install --upgrade   langchain langchain-core langchain-community langchain-classic   langchain-text-splitters langchain-huggingface langchain-groq langchain-cohere   faiss-cpu rank_bm25 sentence-transformers pypdf reportlab tenacity   ragas datasets cohere groq tiktoken

print("installs done")


installs done


##2. Environment & API Configuration
Configures necessary APIs for the pipeline:

Groq: Handles LLM generation (Llama 3 70B) for low-latency inference.

Cohere: Powers the Cross-Encoder reranking model.

The script securely loads keys via Colab Secrets (userdata) with a getpass fallback for portability.

In [ ]:
import os, getpass, warnings, time
warnings.filterwarnings("ignore")

def _load_key(name: str) -> None:
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return
    except Exception:
        pass
    os.environ[name] = getpass.getpass(f"{name} (hidden): ").strip()

_load_key("GROQ_API_KEY")
_load_key("COHERE_API_KEY")

# Optional. MiniLM does not need this. Leave blank.
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

missing = [k for k in ("GROQ_API_KEY", "COHERE_API_KEY") if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"missing keys: {missing}")

print("keys loaded:", "GROQ", "COHERE")


keys loaded: GROQ COHERE


In [ ]:
import json, hashlib, textwrap
from pathlib import Path
from typing import List, Dict, Any, Optional

from tenacity import (
    retry, wait_exponential, stop_after_attempt,
    retry_if_exception_type,
)

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_cohere import CohereRerank

# LangChain 1.x parked EnsembleRetriever / ContextualCompressionRetriever
# under langchain_classic. 0.3 still has them on langchain.retrievers.

try:
    from langchain_classic.retrievers import EnsembleRetriever
    from langchain_classic.retrievers.contextual_compression import (
        ContextualCompressionRetriever,
    )
except ImportError:
    from langchain.retrievers import EnsembleRetriever
    from langchain.retrievers.contextual_compression import (
        ContextualCompressionRetriever,
    )

print("imports ok")


imports ok


## 3. Data Ingestion & Semantic Chunking
This block generates a self-contained synthetic credit agreement and 10-K excerpt for testing.

Architecture Choices:

Custom Overlap (900/180): Standard 50-character overlaps often sever crucial legal definitions (like "Adjusted EBITDA"). A 180-character overlap ensures full clause boundaries are preserved.

Custom Separators: Legal documents are heavily reliant on semicolons. The RecursiveCharacterTextSplitter is customized to split on double newlines, periods, and semicolons to prevent cutting covenants mid-sentence.

Metadata Tagging: Injects section and page metadata directly into the chunks to guarantee accurate inline citations later in the pipeline.

In [ ]:
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER

DOC_PATH = Path("/content/northstar_credit_pack.pdf")
DOC_PATH.parent.mkdir(parents=True, exist_ok=True)


SECTIONS = [
    (
        "cover",
        "Cover",
        """NORTHSTAR CREDIT PARTNERS, L.P. - SENIOR SECURED CREDIT AGREEMENT
dated as of March 14, 2024, among HALE FORGE INDUSTRIES, INC., as Borrower,
the Lenders party hereto, and KEYSTONE ADMINISTRATIVE AGENT LLC, as Agent.
Principal Amount: $425,000,000 Term Loan B. Governing law: State of New York.
This pack also appends excerpts from the Borrower's Form 10-K for the fiscal
year ended December 31, 2025 (Item 1A and Item 7) for RAG evaluation only.""",
    ),
    (
        "defs",
        "Article I - Defined Terms",
        """ "Adjusted EBITDA" means, for any period, Consolidated Net Income of the
Borrower and its Restricted Subsidiaries, plus, without duplication and to
the extent deducted, (a) interest expense, (b) tax expense, (c) depreciation
and amortization, (d) non-cash stock-based compensation, (e) extraordinary
or non-recurring losses not to exceed $12,000,000 in any four-quarter period,
and (f) run-rate cost savings from Permitted Acquisitions projected in good
faith to be realized within 12 months; provided that add-backs under clause
(f) shall not exceed 15% of Adjusted EBITDA before giving effect to such
add-backs. Pro forma adjustments require the Agent's prior written consent
if they exceed $5,000,000 in the aggregate.

"Interest Coverage Ratio" means, as of any Test Date, the ratio of (a)
Adjusted EBITDA for the four fiscal quarter period then ended to (b)
Consolidated Interest Expense for such period.

"Leverage Ratio" means, as of any Test Date, the ratio of (a) Consolidated
Net Debt as of such date to (b) Adjusted EBITDA for the four fiscal quarter
period then ended.

"Material Adverse Effect" means a material adverse effect on (a) the
business, assets, or financial condition of the Borrower and its Restricted
Subsidiaries, taken as a whole, (b) the ability of the Borrower to perform
the payment Obligations, or (c) the validity or enforceability of any Loan
Document or the rights of the Agent or the Lenders thereunder. A Material
Adverse Effect does not include changes in general economic, banking, or
securities market conditions, or changes in GAAP, except to the extent such
changes disproportionately affect the Borrower relative to other participants
in its industry.

"Change of Control" means (a) any Person or group acquiring beneficial
ownership of more than 35% of the voting stock of the Borrower, (b) a sale
of all or substantially all assets of the Borrower, or (c) the Continuing
Directors ceasing to constitute a majority of the Board. A Permitted Holdco
Reorganization is not a Change of Control.

"Restricted Payment" means any dividend or distribution on Equity Interests,
any redemption or repurchase of Equity Interests, any payment on Subordinated
Indebtedness prior to its scheduled maturity, and any payment of management
fees to Affiliates other than under the Management Agreement as in effect
on the Closing Date.

"Excess Cash Flow" means Adjusted EBITDA minus capital expenditures,
scheduled principal payments, cash taxes, and increases in working capital,
plus decreases in working capital, for each Excess Cash Flow Period.""",
    ),
    (
        "covenants",
        "Section 6.08 - Financial Covenants",
        """The Borrower shall not permit the Interest Coverage Ratio as of the last
day of any fiscal quarter (each, a "Test Date"), commencing with the fiscal
quarter ending June 30, 2024, to be less than 2.50 to 1.00. The Interest
Coverage Ratio is tested quarterly, in arrears, on a trailing-four-quarter
basis. A single miss is an Event of Default unless cured by an Equity Cure
under Section 8.03 within 10 Business Days after the date the compliance
certificate is required to be delivered.

The Borrower shall not permit the Leverage Ratio as of any Test Date to
exceed (i) 5.50 to 1.00 through the Test Date on December 31, 2024, (ii)
5.00 to 1.00 through December 31, 2025, and (iii) 4.50 to 1.00 thereafter.
The step-downs are automatic and are not subject to waiver except by
Required Lenders.

Financial covenants are calculated on a pro forma basis for Permitted
Acquisitions and Disposition of assets greater than $10,000,000, as if
such transaction had occurred on the first day of the relevant four-quarter
period. Currency translation uses the average exchange rate for the period
for the income statement and the period-end rate for the balance sheet.""",
    ),
    (
        "rp",
        "Section 6.12 - Restricted Payments",
        """The Borrower shall not declare or make any Restricted Payment, except:
(a) dividends payable solely in Qualified Equity Interests; (b) Restricted
Payments from a Restricted Subsidiary to the Borrower or another Restricted
Subsidiary; (c) so long as no Default or Event of Default exists or would
result, Restricted Payments in an amount not to exceed the Available Amount;
and (d) additional Restricted Payments if the Leverage Ratio, on a pro forma
basis after giving effect thereto, is less than or equal to 3.50 to 1.00,
and the Interest Coverage Ratio is at least 3.00 to 1.00.

Available Amount starts at $25,000,000 on the Closing Date and builds by
50% of Excess Cash Flow not required to be prepaid under Section 2.11(b),
plus Qualified Equity proceeds, minus Restricted Payments made under clause
(c). The Available Amount builder is not available during a Default.

Affiliate management fees under the Management Agreement may be paid only
if no Event of Default has occurred and is continuing, and only up to
$2,000,000 per fiscal year.""",
    ),
    (
        "ecf",
        "Section 2.11 - Mandatory Prepayments",
        """Within 10 Business Days after the annual financial statements are delivered
under Section 5.01, the Borrower shall prepay the Term Loans in an amount
equal to (i) 50% of Excess Cash Flow for such Excess Cash Flow Period if
the Leverage Ratio as of the last day of such period is greater than 4.00
to 1.00, (ii) 25% if the Leverage Ratio is greater than 3.25 to 1.00 but
less than or equal to 4.00 to 1.00, and (iii) 0% if the Leverage Ratio is
less than or equal to 3.25 to 1.00. Each Excess Cash Flow prepayment is
applied in direct order of maturity to the remaining amortization
installments, and then to the balloon.

Asset sale net proceeds greater than $5,000,000 in a single transaction
(or a series of related transactions) must be applied to prepay Term Loans
within 5 Business Days unless reinvested in productive assets used in the
business within 365 days (or 545 days if committed within 365 days).""",
    ),
    (
        "eod",
        "Section 7.01 - Events of Default",
        """Each of the following constitutes an Event of Default: (a) failure to pay
principal when due; (b) failure to pay interest or fees within 3 Business
Days of the due date; (c) breach of Section 6.08 (Financial Covenants),
subject to the Equity Cure; (d) breach of any other covenant that remains
uncured 30 days after notice from the Agent; (e) a Change of Control;
(f) a Material Adverse Effect that is not remedied within 20 days of the
Borrower becoming aware of it, solely with respect to clause (b) of that
definition; (g) cross-default to other Material Indebtedness of $20,000,000
or more; (h) bankruptcy or insolvency proceedings; and (i) a final judgment
in excess of $15,000,000 that remains unstayed for 45 days.

Upon an Event of Default, the Agent may, and at the request of Required
Lenders shall, declare the Obligations due and payable. Automatic
acceleration applies to insolvency Events of Default under clause (h).""",
    ),
    (
        "cure",
        "Section 8.03 - Equity Cure",
        """In the event of a breach of Section 6.08, the Borrower may, not more than
twice in any four-quarter period and not more than four times during the
life of the facility, receive a cash equity contribution that is included
in Adjusted EBITDA for the purpose of recasting the failed Test Date (an
"Equity Cure"). The Equity Cure amount shall not exceed the shortfall
plus a 10% buffer. The Equity Cure must be received within 10 Business Days
after the compliance certificate for that Test Date is due. Equity Cures
are disregarded when calculating Excess Cash Flow and when testing the
Availability Amount builder. An Equity Cure does not cure a payment
default or a Change of Control default.""",
    ),
    (
        "10k-risk",
        "Form 10-K Item 1A - Risk Factors (excerpt)",
        """Our senior secured credit facility contains restrictive covenants, including
a minimum Interest Coverage Ratio of 2.50x and a maximum Leverage Ratio
that steps down from 5.50x to 4.50x. Failure to comply could result in
acceleration of $425 million of term loan principal. We relied on an
Equity Cure in the quarter ended September 30, 2025 after a one-time
inventory write-down compressed Adjusted EBITDA. We cannot assure you that
the lenders will permit additional cures, and the credit agreement caps
cures at four over the life of the facility.

A substantial portion of our debt bears interest at SOFR plus 4.25%. A
100 basis-point increase in SOFR would increase annual interest expense by
approximately $4.3 million, which would pressure the Interest Coverage
Ratio if Adjusted EBITDA does not grow in line.

We are a defendant in Hale v. Hale Forge, a putative class action in the
District of Delaware alleging misstatements in our fiscal 2024 revenue
recognition. We believe the claims are without merit. An unfavorable
outcome could exceed our $8 million accrual and, if above $15 million and
unstayed, could constitute an Event of Default under the credit agreement.""",
    ),
    (
        "10k-mda",
        "Form 10-K Item 7 - MD&A (excerpt)",
        """Liquidity. As of December 31, 2025 we had $41 million of cash and $80 million
undrawn on the revolving credit facility, which remains undrawn since
close. The Term Loan B amortizes at 1% per annum with a balloon at maturity
on March 14, 2031. Excess Cash Flow sweep for fiscal 2025 was 25% because
our year-end Leverage Ratio was 3.74x, resulting in a $11.6 million
prepayment applied to remaining amortization.

Covenant headroom. The Interest Coverage Ratio was 3.12x at year-end 2025
versus a 2.50x floor. The Leverage Ratio was 3.74x versus a 5.00x cap for
that Test Date. Management expects the 4.50x leverage cap beginning in
2026 to remain achievable if the Aurora stamping acquisition contributes
at least $18 million of run-rate Adjusted EBITDA, which is within the 15%
add-back cap.

We paid no cash dividends in 2025. Restricted Payments under the Available
Amount basket were $4.1 million of share withholdings for tax on vested
equity awards. The Available Amount at year-end was $33.8 million.""",
    ),
]


def write_sample_pdf(path: Path) -> None:
    styles = getSampleStyleSheet()
    title = ParagraphStyle(
        "DocTitle",
        parent=styles["Heading1"],
        fontName="Times-Bold",
        fontSize=14,
        leading=18,
        alignment=TA_CENTER,
        spaceAfter=16,
    )
    h = ParagraphStyle(
        "SecHead",
        parent=styles["Heading2"],
        fontName="Times-Bold",
        fontSize=12,
        leading=16,
        spaceBefore=10,
        spaceAfter=8,
    )
    body = ParagraphStyle(
        "BodyJust",
        parent=styles["Normal"],
        fontName="Times-Roman",
        fontSize=10,
        leading=14,
        alignment=TA_JUSTIFY,
        spaceAfter=8,
    )
    doc = SimpleDocTemplate(
        str(path),
        pagesize=LETTER,
        leftMargin=0.85 * inch,
        rightMargin=0.85 * inch,
        topMargin=0.8 * inch,
        bottomMargin=0.8 * inch,
        title="Northstar Credit Pack",
        author="Citebound sample",
    )
    story = []
    story.append(Paragraph("NORTHSTAR CREDIT PACK (SAMPLE)", title))
    story.append(Paragraph(
        "Confidential sample for retrieval evaluation. Not an offer to sell securities.",
        body,
    ))
    for i, (_key, heading, text) in enumerate(SECTIONS):
        if i in (1, 7):
            story.append(PageBreak())
        story.append(Paragraph(heading, h))
        clean = (
            text.strip()
            .replace("&", "&amp;")
            .replace('"', "&quot;")
            .replace("\n\n", "<br/><br/>")
            .replace("\n", " ")
        )
        story.append(Paragraph(clean, body))
        story.append(Spacer(1, 8))
    doc.build(story)
    print("wrote", path, "bytes", path.stat().st_size)

write_sample_pdf(DOC_PATH)


wrote /content/northstar_credit_pack.pdf bytes 10974


In [ ]:
# PyPDFLoader used for lightweight in-memory metadata retention.
loader = PyPDFLoader(str(DOC_PATH))
raw_pages = loader.load()
print(f"pages: {len(raw_pages)}")
print("page-0 preview:", raw_pages[0].page_content[:240].replace("\n", " "))

# 900/180 chunk/overlap ratio prevents truncation of complex legal clauses.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
    length_function=len,
)

chunks = splitter.split_documents(raw_pages)

def _guess_section(text: str) -> str:
    head = text[:180].lower()
    if "adjusted ebitda" in head or "defined term" in head or "means," in head:
        return "defined-terms"
    if "financial covenant" in head or "interest coverage" in head and "shall not permit" in head:
        return "financial-covenants"
    if "restricted payment" in head:
        return "restricted-payments"
    if "excess cash flow" in head and "prepay" in head:
        return "mandatory-prepayments"
    if "event of default" in head:
        return "events-of-default"
    if "equity cure" in head:
        return "equity-cure"
    if "risk factor" in head or "putative class" in head:
        return "10k-risk-factors"
    if "liquidity" in head or "covenant headroom" in head:
        return "10k-mda"
    return "general"

for i, ch in enumerate(chunks):
    ch.metadata["chunk_id"] = i
    ch.metadata["section"] = _guess_section(ch.page_content)
    ch.metadata["doc_type"] = "credit-agreement" if ch.metadata.get("page", 0) < 6 else "10-k"
    # stable id so we can cite the same chunk across rerank vs eval
    ch.metadata["uid"] = hashlib.md5(ch.page_content.encode("utf-8")).hexdigest()[:10]

print(f"chunks: {len(chunks)}")
from collections import Counter
print("sections:", dict(Counter(c.metadata["section"] for c in chunks)))


pages: 5
page-0 preview: NORTHSTAR CREDIT PACK (SAMPLE) Confidential sample for retrieval evaluation. Not an offer to sell securities. Cover NORTHSTAR CREDIT PARTNERS, L.P. - SENIOR SECURED CREDIT AGREEMENT dated as of March 14, 2024, among HALE FORGE INDUSTRIES, I
chunks: 15
sections: {'general': 5, 'defined-terms': 3, 'events-of-default': 3, 'mandatory-prepayments': 2, 'equity-cure': 1, '10k-risk-factors': 1}


## 4. Hybrid Search Retriever
Combines sparse keyword search and dense semantic search to eliminate single-mode blind spots:
- **BM25 (Sparse):** Exact match for section IDs, numerical thresholds, and proprietary terms.
- **FAISS (Dense):** Conceptual semantic matching via `sentence-transformers/all-MiniLM-L6-v2`.
- **Ensemble:** 50/50 weighted rank fusion pulling top 15 candidate chunks.

In [ ]:


import torch

embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # cosine via inner product
)


vectordb = FAISS.from_documents(chunks, embed_model)
dense = vectordb.as_retriever(search_kwargs={"k": 15})

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 15

hybrid = EnsembleRetriever(
    retrievers=[bm25, dense],
    weights=[0.5, 0.5],
)

def _show(docs: List[Document], n: int = 5) -> None:
    for i, d in enumerate(docs[:n], 1):
        snip = d.page_content.replace("\n", " ")[:160]
        print(f"""{i:02d}  p.{d.metadata.get('page','?')}  {d.metadata.get('section')}\n    {snip}...\n""")

probe = "What is the Interest Coverage Ratio and when is it tested?"
print("BM25 only:")
_show(bm25.invoke(probe))
print("FAISS only:")
_show(dense.invoke(probe))
print("hybrid (RRF):")
_show(hybrid.invoke(probe))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25 only:
01  p.3  equity-cure
    Section 8.03 - Equity Cure In the event of a breach of Section 6.08, the Borrower may, not more than twice in any four-quarter period and not more than four tim...

02  p.2  general
    after giving effect thereto, is less than or equal to 3.50 to 1.00, and the Interest Coverage Ratio is at least 3.00 to 1.00. Available Amount starts at $25,000...

03  p.1  general
    Interests, any payment on Subordinated Indebtedness prior to its scheduled maturity, and any payment of management fees to Affiliates other than under the Manag...

04  p.2  general
    than $10,000,000, as if such transaction had occurred on the first day of the relevant four-quarter period. Currency translation uses the average exchange rate ...

05  p.2  mandatory-prepayments
    prepay the Term Loans in an amount equal to (i) 50% of Excess Cash Flow for such Excess Cash Flow Period if the Leverage Ratio as of the last day of such period...

FAISS only:
01  p.1  general
    Interest

## 5. Reranking Layer
Applies a cross-encoder re-scoring pass using `CohereRerank` wrapped in a `ContextualCompressionRetriever`.
- Compresses top 15 hybrid search results down to the top 4 highest-relevance chunks.
- Minimizes context window noise and reduces downstream token latency.

In [ ]:
import cohere

class Cohere429(Exception):
    pass

# langchain_cohere's CohereRerank already talks to the API; we still wrap
# the retriever invoke so a transient 429 does not kill an eval loop.
reranker = CohereRerank(
    model="rerank-english-v3.0",
    top_n=4,
    cohere_api_key=os.environ["COHERE_API_KEY"],
)

compressed = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid,
)

@retry(
    wait=wait_exponential(multiplier=1, min=1, max=20),
    stop=stop_after_attempt(4),
    reraise=True,
)
def retrieve(query: str) -> List[Document]:
    try:
        docs = compressed.invoke(query)
    except Exception as e:
        msg = str(e).lower()
        if "429" in msg or "rate" in msg:
            raise Cohere429(str(e))
        raise
    return docs

reranked = retrieve(probe)
print(f"hybrid → rerank: {15} candidates considered, {len(reranked)} kept\n")
_show(reranked, n=4)
# relevance_score is attached by the compressor when Cohere returns it
for d in reranked:
    print("score", round(float(d.metadata.get("relevance_score", -1)), 3),
          "uid", d.metadata.get("uid"))


hybrid → rerank: 15 candidates considered, 4 kept

01  p.1  defined-terms
    they exceed $5,000,000 in the aggregate. "Interest Coverage Ratio" means, as of any Test Date, the ratio of (a) Adjusted EBITDA for the four fiscal quarter peri...

02  p.1  general
    Interests, any payment on Subordinated Indebtedness prior to its scheduled maturity, and any payment of management fees to Affiliates other than under the Manag...

03  p.4  mandatory-prepayments
    on March 14, 2031. Excess Cash Flow sweep for fiscal 2025 was 25% because our year-end Leverage Ratio was 3.74x, resulting in a $11.6 million prepayment applied...

04  p.1  defined-terms
    Article I - Defined Terms "Adjusted EBITDA" means, for any period, Consolidated Net Income of the Borrower and its Restricted Subsidiaries, plus, without duplic...

score 0.998 uid 00628e94ee
score 0.997 uid d8b57416b9
score 0.938 uid 84ff923637
score 0.882 uid ff30960223


## 6. Generation & Citation Guardrails
Configures `ChatGroq` (qwen/qwen3.8-27b) with a strict grounding prompt template:
- Forces structured inline attribution referencing chunk metadata (`[section, page]`).
- Implements an explicit fallback response ("I do not have enough context") if retrieved context lacks the answer.

In [ ]:
SYSTEM = """You are a capital-markets analyst answering from excerpts of a credit
agreement and a 10-K. You are not a lawyer. You are not allowed to be helpful
beyond the excerpts.

Rules you cannot break:
- Use only the excerpts inside <sources>. If they do not contain the answer,
  reply with exactly: I do not have enough context.
- After every sentence that states a fact, put an inline citation like [S1]
  or [S1][S3] matching the source tags. No sentence with a number, a date,
  a ratio, or a defined term ships without a tag.
- Quote defined terms exactly, including capitalisation. Do not rename
  "Interest Coverage Ratio" to "interest cover".
- If two excerpts disagree, say they disagree and cite both. Do not pick a winner.
- Do not use outside knowledge of the Credit Agreement market, even if it is
  "standard". Standard is how hallucinations sneak in.
- No preamble ("Based on the provided context"). No sign-off.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human",
     "Question: {question}\n\n"
     "<sources>\n{context}\n</sources>\n\n"
     "Answer with citations."),
])

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=1024,
    groq_api_key=os.environ["GROQ_API_KEY"],
)

def format_sources(docs: List[Document]) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        sec = d.metadata.get("section", "general")
        uid = d.metadata.get("uid", "")
        blocks.append(
            f"[S{i}] section={sec} page={page} id={uid}\n{d.page_content.strip()}"
        )
    return "\n\n".join(blocks) if blocks else "(no sources)"

@retry(
    wait=wait_exponential(multiplier=1, min=1, max=16),
    stop=stop_after_attempt(4),
    reraise=True,
)
def _generate(question: str, context: str) -> str:
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question, "context": context})

def ask(question: str, verbose: bool = True) -> Dict[str, Any]:
    t0 = time.time()
    docs = retrieve(question)
    ctx = format_sources(docs)
    answer = _generate(question, ctx)
    took = time.time() - t0
    payload = {
        "question": question,
        "answer": answer.strip(),
        "sources": [
            {
                "tag": f"S{i}",
                "section": d.metadata.get("section"),
                "page": d.metadata.get("page"),
                "uid": d.metadata.get("uid"),
                "score": d.metadata.get("relevance_score"),
                "text": d.page_content,
            }
            for i, d in enumerate(docs, 1)
        ],
        "latency_s": round(took, 2),
    }
    if verbose:
        print(f"""Q: {question}
({payload['latency_s']}s, {len(docs)} chunks after rerank)

{payload['answer']}
""")
        print("--- sources ---")
        for s in payload["sources"]:
            print(f"  [{s['tag']}] {s['section']} p.{s['page']} uid={s['uid']}")
        print()
    return payload

# Sanity: this should cite Section 6.08 / defined terms, not invent a 3.0x floor.
out_ok = ask("What is the Interest Coverage Ratio covenant and when is it tested?")

# Sanity: the corpus has no CEO net worth. The model must refuse.
out_refuse = ask("What is the CEO's personal net worth according to the credit agreement?")

Q: What is the Interest Coverage Ratio covenant and when is it tested?
(0.55s, 4 chunks after rerank)

The "Interest Coverage Ratio" is defined as the ratio of (a) Adjusted EBITDA for the four fiscal quarter period then ended to (b) Consolidated Interest Expense for such period [S2]. The Borrower is required to maintain an "Interest Coverage Ratio" of at least 2.50 to 1.00 [S1]. This covenant is tested quarterly, in arrears, on a trailing-four-quarter basis [S1]. The testing commences with the fiscal quarter ending June 30, 2024 [S1].

--- sources ---
  [S1] general p.1 uid=d8b57416b9
  [S2] defined-terms p.1 uid=00628e94ee
  [S3] mandatory-prepayments p.4 uid=84ff923637
  [S4] 10k-risk-factors p.4 uid=6f3de710d1

Q: What is the CEO's personal net worth according to the credit agreement?
(0.62s, 4 chunks after rerank)

I do not have enough context.

--- sources ---
  [S1] general p.2 uid=37e6e87c7e
  [S2] defined-terms p.1 uid=00628e94ee
  [S3] defined-terms p.4 uid=fbc58a7ee1
  [S4] 1

In [ ]:
# One more, slightly messier — Restricted Payments is a place models like to
# "helpfully" merge the Available Amount basket with the leverage gate.
out_rp = ask(
    "Under what conditions can Hale Forge pay a cash dividend, and what is "
    "the Available Amount on the Closing Date?"
)

class CiteboundAnalyzer:
    """Thin wrapper so the rest of a service can depend on a class, not a pile
    of notebook globals. Same objects — we are not rebuilding FAISS."""

    def __init__(self, retriever, generate_fn):
        self.retriever = retriever
        self._generate_fn = generate_fn

    def retrieve(self, question: str) -> List[Document]:
        return retrieve(question)

    def ask(self, question: str, verbose: bool = False) -> Dict[str, Any]:
        return ask(question, verbose=verbose)

analyzer = CiteboundAnalyzer(compressed, _generate)
print("analyzer ready")


Q: Under what conditions can Hale Forge pay a cash dividend, and what is the Available Amount on the Closing Date?
(0.44s, 4 chunks after rerank)

Hale Forge can pay a cash dividend if, after giving effect to the payment, the Leverage Ratio is less than or equal to 3.50 to 1.00 and the Interest Coverage Ratio is at least 3.00 to 1.00 [S4]. The Available Amount on the Closing Date is $25,000,000 [S4].

--- sources ---
  [S1] defined-terms p.4 uid=fbc58a7ee1
  [S2] general p.0 uid=4295cb2ddd
  [S3] mandatory-prepayments p.4 uid=84ff923637
  [S4] general p.2 uid=bebe41bb67

analyzer ready


## 7. Automated Evaluation (Ragas)
Executes quantitative evaluation pipeline against ground-truth QA pairs:
- **Context Precision:** Evaluates signal-to-noise ratio of retrieved chunks.
- **Faithfulness:** Verifies zero-hallucination compliance against provided context.
- **Answer Relevancy:** Measures semantic alignment with user intent.
- Exports evaluation dataframe to `/content/citebound_ragas.csv`.



In [ ]:
from datasets import Dataset

EVAL_SET = [
    {
        "question": "What is the minimum Interest Coverage Ratio and how often is it tested?",
        "ground_truth": (
            "The Borrower shall not permit the Interest Coverage Ratio to be less "
            "than 2.50 to 1.00. It is tested quarterly, in arrears, on a "
            "trailing-four-quarter basis as of each Test Date, commencing with "
            "the fiscal quarter ending June 30, 2024."
        ),
    },
    {
        "question": "When is a Change of Control an Event of Default, and what ownership threshold triggers Change of Control?",
        "ground_truth": (
            "A Change of Control is an Event of Default under Section 7.01(e). "
            "Change of Control includes any Person or group acquiring beneficial "
            "ownership of more than 35% of the voting stock of the Borrower. "
            "A Permitted Holdco Reorganization is not a Change of Control."
        ),
    },
    {
        "question": "What Excess Cash Flow sweep percentage applied for fiscal 2025 and why?",
        "ground_truth": (
            "The Excess Cash Flow sweep for fiscal 2025 was 25% because the "
            "year-end Leverage Ratio was 3.74x, which sits in the band greater "
            "than 3.25x and less than or equal to 4.00x. That produced an "
            "$11.6 million prepayment."
        ),
    },
]

rows = []
for item in EVAL_SET:
    payload = analyzer.ask(item["question"], verbose=False)
    rows.append({
        "question": item["question"],
        "user_input": item["question"],
        "answer": payload["answer"],
        "response": payload["answer"],
        "contexts": [s["text"] for s in payload["sources"]],
        "retrieved_contexts": [s["text"] for s in payload["sources"]],
        "ground_truth": item["ground_truth"],
        "reference": item["ground_truth"],
    })
    print("collected", item["question"][:64], "... →", payload["answer"][:72], "...")

eval_ds = Dataset.from_list(rows)
print("eval rows:", len(eval_ds))


collected What is the minimum Interest Coverage Ratio and how often is it  ... → The minimum Interest Coverage Ratio is 2.50 to 1.00 [S2]. This requireme ...
collected When is a Change of Control an Event of Default, and what owners ... → A Change of Control constitutes an Event of Default under clause (e) of  ...
collected What Excess Cash Flow sweep percentage applied for fiscal 2025 a ... → The Excess Cash Flow sweep percentage applied for fiscal 2025 was 25% [S ...
eval rows: 3


In [ ]:

# Wrap LangChain judge models into Ragas-compatible interfaces
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm_raw = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"],
)
judge_llm = LangchainLLMWrapper(judge_llm_raw)
judge_emb = LangchainEmbeddingsWrapper(embed_model)

def _run_ragas(ds):
    # Prefer the 0.3 class API, fall back to the older function objects.
    try:
        from ragas.metrics import (
            Faithfulness,
            AnswerRelevancy,
            LLMContextPrecisionWithReference,
        )
        metrics = [
            Faithfulness(),
            AnswerRelevancy(),
            LLMContextPrecisionWithReference(),
        ]
        print("ragas metric API: classes (0.3+)")
    except Exception:
        from ragas.metrics import faithfulness, answer_relevancy, context_precision
        metrics = [faithfulness, answer_relevancy, context_precision]
        print("ragas metric API: functions (0.2)")

    return evaluate(
        dataset=ds,
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_emb,
        raise_exceptions=False,
    )

result = _run_ragas(eval_ds)
df = result.to_pandas()

# Keep a readable subset — ragas dumps extra columns that hide the point.
keep = [c for c in df.columns if c in (
    "question", "user_input", "answer", "response",
    "faithfulness", "answer_relevancy", "answer_relevance",
    "context_precision", "llm_context_precision_with_reference",
)]
view = df[keep] if keep else df
print("\n=== Ragas evaluation ===")
display(view)

num_cols = view.select_dtypes(include="number")
if len(num_cols.columns):
    print("\nmeans:")
    print(num_cols.mean(numeric_only=True).round(3))

df.to_csv("/content/citebound_ragas.csv", index=False)
print("\nwrote /content/citebound_ragas.csv")

ragas metric API: classes (0.3+)


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[2]: NotFoundError(Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}})
ERROR:ragas.executor:Exception raised in Job[7]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[8]: NotFoundError(Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}})
ERROR:ragas.executor:Exception raised in Job[4]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invali


=== Ragas evaluation ===


,user_input,response,faithfulness,answer_relevancy,llm_context_precision_with_reference
0,What is the minimum Interest Coverage Ratio an...,The minimum Interest Coverage Ratio is 2.50 to...,1.0,0.928887,0.500000
1,When is a Change of Control an Event of Defaul...,A Change of Control constitutes an Event of De...,1.0,0.810053,0.500000
2,What Excess Cash Flow sweep percentage applied...,The Excess Cash Flow sweep percentage applied ...,1.0,0.966224,0.916667



means:
faithfulness                            1.000
answer_relevancy                        0.902
llm_context_precision_with_reference    0.639
dtype: float64

wrote /content/citebound_ragas.csv
